In [23]:
import ast
import json
import warnings

import pandas as pd
from snowflake.ml.modeling.impute import SimpleImputer
from snowflake.ml.modeling.metrics import accuracy_score
from snowflake.ml.modeling.model_selection import GridSearchCV
from snowflake.ml.modeling.preprocessing import OrdinalEncoder#OneHotEncoder
from snowflake.ml.modeling.xgboost import XGBClassifier
from snowflake.ml.registry import Registry
from snowflake.ml.model import type_hints

from snowflake.snowpark import Session
from snowflake.snowpark import types as T
from snowflake.snowpark.functions import col
from snowflake.ml.feature_store import (
    FeatureStore,
    FeatureView,
    Entity,
    CreationMode
)
warnings.simplefilter(action="ignore", category=UserWarning)

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Load Data
![](https://github.com/sheena-n/Snowflake-Champion-Challenger/blob/dev/images/champion_challenger_flow.png)
![](https://raw.githubusercontent.com/Snowflake-Labs/snowflake-demo-notebooks/main/Navigating%20and%20Browsing%20Files/img/git_files.png)

In [34]:
customer_df = session.table("customer_churn")
customer_df = customer_df.drop(['CREATED_DT', 'CUSTOMER_NAME', 'LAST_ORDER_DT', 'FIRST_ORDER_DT','CITY'])
customer_df.show()

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"CUSTOMER_ID"  |"STATE"  |"FAV_DELIVERY_DAY"  |"REFILL"  |"DOOR_DELIVERY"  |"PAPERLESS"  |"RETAINED"  |"ESENT"  |"EOPENRATE"  |"ECLICKRATE"  |"AVG_ORDER"  |"DIFF_BETWEEN_LAST_FIRST_DAYS"  |"DIFF_BETWEEN_FIRST_CREATED_DAYS"  |
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|6H6T6N         |TX       |Monday              |0         |0                |0            |0           |29       |100.0        |3.448275862   |5.32000      |0                               |317                                |
|APCENR         |TX       |Friday              |1         |1                |1            |1

In [36]:
cat_cols = ["STATE", "FAV_DELIVERY_DAY"]
cat_cols_en = ["STATE_EN", "FAV_DELIVERY_DAY_EN"]
num_cols = ["REFILL", "DOOR_DELIVERY", "PAPERLESS", "RETAINED", "ESENT","EOPENRATE","ECLICKRATE","AVG_ORDER","DIFF_BETWEEN_LAST_FIRST_DAYS","DIFF_BETWEEN_FIRST_CREATED_DAYS"]

# Feature Engineering

In [37]:
impute_cat = SimpleImputer(
    input_cols=cat_cols,
    output_cols=cat_cols,
    strategy="most_frequent",
    drop_input_cols=True,
)
customer_df = impute_cat.fit(customer_df).transform(customer_df)
customer_df.show()

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"STATE"  |"FAV_DELIVERY_DAY"  |"CUSTOMER_ID"  |"REFILL"  |"DOOR_DELIVERY"  |"PAPERLESS"  |"RETAINED"  |"ESENT"  |"EOPENRATE"  |"ECLICKRATE"  |"AVG_ORDER"  |"DIFF_BETWEEN_LAST_FIRST_DAYS"  |"DIFF_BETWEEN_FIRST_CREATED_DAYS"  |
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|TX       |Monday              |6H6T6N         |0         |0                |0            |0           |29       |100.0        |3.448275862   |5.32000      |0                               |317                                |
|TX       |Friday              |APCENR         |1         |1                |1            |1

In [38]:
OE = OrdinalEncoder(
    input_cols=cat_cols,
    output_cols=cat_cols_en
)

customer_df = OE.fit(customer_df).transform(customer_df)

customer_df = customer_df.drop(cat_cols)
customer_df.show()

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"STATE"  |"FAV_DELIVERY_DAY"  |"CUSTOMER_ID"  |"REFILL"  |"DOOR_DELIVERY"  |"PAPERLESS"  |"RETAINED"  |"ESENT"  |"EOPENRATE"  |"ECLICKRATE"  |"AVG_ORDER"  |"DIFF_BETWEEN_LAST_FIRST_DAYS"  |"DIFF_BETWEEN_FIRST_CREATED_DAYS"  |
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|0.0      |1.0                 |6H6T6N         |0         |0                |0            |0           |29       |100.0        |3.448275862   |5.32000      |0                               |317                                |
|0.0      |0.0                 |APCENR         |1         |1                |1            |1

In [39]:
train_df, test_df = customer_df.random_split(weights=[0.8, 0.2], seed=8)

In [11]:
parameters = {
    "n_estimators": [100, 200, 300, 400, 500, 600],
    "learning_rate": [0.1, 0.2, 0.3, 0.4, 0.5],
    "max_depth": list(range(3, 6, 1)),
    "min_child_weight": list(range(1, 6, 1)),
}

In [13]:
session.sql(
    f"ALTER WAREHOUSE {session.get_current_warehouse()[1:-1]} SET WAREHOUSE_SIZE=XLARGE;"
).collect()

[Row(status='Statement executed successfully.')]

In [14]:
grid_search = GridSearchCV(
    estimator=XGBClassifier(),
    param_grid=parameters,
    n_jobs=-1,
    scoring="accuracy",
    input_cols=train_df.drop(["RETAINED","CUSTOMER_ID"]).columns,
    label_cols="RETAINED",
    output_cols="PRED_RETAINED",
)

# Train
grid_search.fit(train_df)

The version of package 'scikit-learn' in the local environment is 1.3.2, which does not fit the criteria for the requirement 'scikit-learn==1.3.0'. Your UDF might not work when the package version is different between the server and your local environment.
Package 'fastparquet' is not installed in the local environment. Your UDF might not work when the package is installed on the server but not on your local environment.
The version of package 'pyarrow' in the local environment is 16.1.0, which does not fit the criteria for the requirement 'pyarrow<14'. Your UDF might not work when the package version is different between the server and your local environment.


In [15]:
#If you don't have permission to alter size of WH please go for "USE WAREHOUSE .." to switch to bigger ones.
session.sql(
    f"ALTER WAREHOUSE {session.get_current_warehouse()[1:-1]} SET WAREHOUSE_SIZE=XSMALL;"
).collect()

[Row(status='Statement executed successfully.')]

In [16]:
result = grid_search.predict(test_df)

The version of package 'scikit-learn' in the local environment is 1.3.2, which does not fit the criteria for the requirement 'scikit-learn==1.3.0'. Your UDF might not work when the package version is different between the server and your local environment.


In [17]:
from snowflake.ml.modeling import metrics
import numpy as np
accuracy = accuracy_score(df=result, y_true_col_names="RETAINED", y_pred_col_names="PRED_RETAINED")
f1_score = metrics.f1_score(df = result,y_true_col_names="RETAINED", y_pred_col_names="PRED_RETAINED")

f1_score = np.round(f1_score,2)
accuracy = np.round(accuracy,2)
print(f"F1 Score: {f1_score}")
print(f"Accuracy: {accuracy}")

Accuracy: 0.962175


In [18]:
# Print each combination of hyperparameters with their accuracy
results = grid_search.to_sklearn().cv_results_
data = {"accuracy": results["mean_test_score"]}
for i, param in enumerate(results["params"]):
    for key, value in param.items():
        if key not in data:
            data[key] = [None] * len(results["params"])
        data[key][i] = value

# Create DataFrame
hp_df = pd.DataFrame(data).sort_values(by="accuracy", ascending=False)
hp_df.head()

,accuracy,learning_rate,max_depth,min_child_weight,n_estimators
66,0.955319,0.1,5,2,100
60,0.954832,0.1,5,1,100
49,0.954711,0.1,4,4,200
127,0.954629,0.2,4,2,200
72,0.954589,0.1,5,3,100


# Model Registry


In [19]:
optimal_model = grid_search.to_sklearn().best_estimator_

In [ ]:
# X = train_df.drop(["RETAINED",'CUSTOMER_ID']).limit(100)

# # session.sql('use database prod_customer_data')
# # session.sql('use schema churn')
# # Create a registry and log the model
# reg = Registry(session=session, database_name='prod_customer_data', schema_name='churn')

# reg_df = reg.show_models()

In [ ]:

USE ROLE DEV_ML_ADMIN;

In [22]:
# Get sample input data to pass into the registry logging function
X = train_df.drop(["RETAINED",'CUSTOMER_ID']).limit(100)

# Create a registry and log the model
reg = Registry(session=session)

reg_df = reg.show_models()

# Define model name and version (use uppercase for name)
model_name = "CHURN"

metrics_dict = {
    "accuracy": accuracy,
    "f1_score": f1_score}

churn_model = reg.log_model(
    model_name=model_name,
    #version_name = 'V_6',
    model=optimal_model,
    comment="Random Forest: Model trained using GridsearchCV",
    sample_input_data=X,
    options={'relax_version': False},
    metrics=metrics_dict,
    task=type_hints.Task.TABULAR_BINARY_CLASSIFICATION
)
    


In [ ]:
-- CREATE TAG live_version COMMENT = 'live version identification tag';

In [ ]:
churn_model.unset_alias("LIVE")

In [ ]:
version_name = churn_model.version_name
churn_model.set_alias("LIVE")
model = reg.get_model(model_name)
model.set_tag("live_version", version_name)

In [ ]:
print("This model has been registered as: " + version_name)

In [ ]:
remote_prediction = churn_model.run(test_df, function_name="predict_proba")
remote_prediction.show()

-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"STATE"  |"FAV_DELIVERY_DAY"  |"REFILL"  |"DOOR_DELIVERY"  |"PAPERLESS"  |"RETAINED"  |"ESENT"  |"EOPENRATE"  |"ECLICKRATE"  |"AVG_ORDER"  |"DIFF_BETWEEN_LAST_FIRST_DAYS"  |"DIFF_BETWEEN_FIRST_CREATED_DAYS"  |"output_feature_0"     |"output_feature_1"      |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|0.0      |6.0                 |1         |0                |1            |1           |46       |80.43478261  |15.2173913    |175.10000    |1190                            |694                                |0.00323200

In [40]:
# To test in SQL write test data back to a table
test_df.write.mode("overwrite").save_as_table("TEST_DATA")
train_df.write.mode("overwrite").save_as_table("TRAIN_DATA")

In [ ]:
hyperparameters = {
    k: v for k, v in optimal_model.get_params().items() if v and k != "missing"
}
churn_model.set_metric(metric_name="hyperparameters", value=hyperparameters)

# code to deploy best model auto
reg_df = reg.get_model(model_name).show_versions()
reg_df["accuracy"] = reg_df["metadata"].apply(
    lambda x: json.loads(x)["metrics"]["accuracy"]
)
best_model = reg_df.sort_values(by="accuracy", ascending=False)